### Documentação das tabelas

Existência de tabelas no Unity Catalog

![](path)
![image_1790561474102.png](./image_1790561474102.png "image_1790561474102.png")

Tabela Silver

![image_1790561592318.png](./image_1790561592318.png "image_1790561592318.png")

Tabela Gold

![image_1790561945474.png](./image_1790561945474.png "image_1790561945474.png")

Comentário adicionado à documentação das tabelas, indicando o que cada campo é

In [0]:
%sql

COMMENT ON TABLE workspace.sinan.silver_sinan IS
'Camada Silver: notificações do SINAN (DATASUS), 2008 em diante, para os agravos coletados (Dengue, Chikungunya, Zika, Tuberculose, Hanseníase, Meningite, Acidente de Trabalho c/ Material Biológico, Violência, Hepatites, Leptospirose, Malária, Câncer, Dermatose, Intoxicação Exógena, PAIR, Pneumoconiose, Tétano Acidental, Transtorno Mental), já com códigos decodificados e campos derivados (ano, geografia, CID, sexo, raça, CAT). Uma linha por notificação. Fonte: FTP ftp.datasus.gov.br, diretórios FINAIS e PRELIM. Renomeada de bronze_sinan_consolidado: a decodificação foi antecipada para dentro do parsing por arquivo por restrição de memória do ambiente serverless, então esta tabela já chega parcialmente conformada, função própria da camada Silver.';

ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN ANO_NOTIF COMMENT 'Ano da notificação, extraído de NU_ANO ou de DT_NOTIFIC quando o primeiro não existe na ficha.';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN COD_REGIAO COMMENT 'Primeiro dígito do código IBGE do município de notificação (1=Norte...5=Centro-Oeste).';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN COD_UF_NOTIF COMMENT 'Dois primeiros dígitos do código IBGE do município de notificação (código da UF).';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN UF_NOTIF COMMENT 'Mesmo valor de COD_UF_NOTIF; UF onde a notificação foi registrada.';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN COD_MUN_NOTIF COMMENT 'Código IBGE completo (6 dígitos) do município de notificação (ID_MUNICIP da ficha original - nunca substituído pelo município de residência).';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN CID10 COMMENT 'Código CID-10 do agravo notificado (campo ID_AGRAVO ou CID da ficha original), maiúsculo, sem tratamento adicional.';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN CID_CAPITULO COMMENT 'Primeira letra do CID10 (aproximação simples; o capítulo oficial correto é calculado na camada Gold, em CAPITULO_CID_OFICIAL).';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN SEXO COMMENT 'Sexo do paciente: Masculino, Feminino ou Ignorado (campo CS_SEXO).';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN RACA COMMENT 'Raça/cor do paciente conforme CS_RACA: Branca, Preta, Amarela, Parda, Indígena ou Ignorado.';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN EMITIA_CAT COMMENT 'Se a ficha emitiu Comunicação de Acidente de Trabalho (Sim/Não/Não se aplica/Ign-Branco), a partir do campo CAT. "Não presente" quando a ficha daquele agravo não tem esse campo.';
ALTER TABLE workspace.sinan.silver_sinan ALTER COLUMN AGRAVO COMMENT 'Nome do agravo/doença conforme classificado neste projeto (ex.: DENGUE, ACIDTRAB) - coluna de partição da tabela.';


COMMENT ON TABLE workspace.sinan.gold_sinan_ntep_detalhe IS
'Camada Gold (nível notificação): Silver enriquecida com a classificação NTEP e filtrada para manter apenas notificações cujo CID corresponde a alguma categoria do Anexo II do Decreto 6.042/2007. Particionada por CATEGORIA_NTEP. É a fonte única de onde as tabelas de fato e dimensão (dim_notificante, dim_local_notificacao, dim_notificacao, fato_notificacoes_ntep) são derivadas.';

ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN ANO_NOTIF COMMENT 'Ano da notificação, herdado da Silver (NU_ANO ou ano de DT_NOTIFIC).';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN COD_REGIAO COMMENT 'Primeiro dígito do código IBGE do município de notificação (1=Norte...5=Centro-Oeste), herdado da Silver. Usado apenas para derivar REGIAO_NOTIF; prefira REGIAO_NOTIF em análises.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN COD_UF_NOTIF COMMENT 'Dois primeiros dígitos do código IBGE do município de notificação (código da UF), herdado da Silver.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN UF_NOTIF COMMENT 'Mesmo valor de COD_UF_NOTIF; UF onde a notificação foi registrada, herdado da Silver.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN COD_MUN_NOTIF COMMENT 'Código IBGE completo (6 dígitos) do município de notificação, herdado da Silver (ID_MUNICIP da ficha original).';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN CID10 COMMENT 'Código CID-10 original da notificação (ID_AGRAVO ou CID da ficha), herdado da Silver sem tratamento adicional - use CID_LIMPO para comparações/joins.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN CID_CAPITULO COMMENT 'Primeira letra do CID10 (aproximação simples, herdada da Silver); o capítulo oficial correto está em CAPITULO_CID_OFICIAL.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN SEXO COMMENT 'Sexo do paciente: Masculino, Feminino ou Ignorado, herdado da Silver.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN RACA COMMENT 'Raça/cor do paciente: Branca, Preta, Amarela, Parda, Indígena ou Ignorado, herdado da Silver.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN EMITIA_CAT COMMENT 'Se a ficha emitiu Comunicação de Acidente de Trabalho (Sim/Não/Não se aplica/Ign-Branco), herdado da Silver. "Não presente" quando a ficha daquele agravo não tem esse campo.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN AGRAVO COMMENT 'Nome do agravo/doença conforme classificado neste projeto (ex.: DENGUE, ACIDTRAB), herdado da Silver. Usado também como critério de desempate na classificação de CATEGORIA_NTEP.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN CID_LIMPO COMMENT 'CID10 normalizado para fins de classificação: maiúsculo e sem ponto (ex.: "A90" a partir de "A90", "C450" a partir de "C45.0"). Coluna intermediária usada para casar com os padrões do dicionário NTEP (exato/prefixo/intervalo); não é a coluna de referência para leitura - use CID10.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN CATEGORIA_NTEP COMMENT 'Categoria do NTEP (do dicionário sinan_ntep_dict.json) a que o CID desta notificação corresponde, com AGRAVO como desempate para CIDs ambíguos entre categorias (ex.: B15-B19 em Hepatite vs. Acidente de Trabalho c/ Material Biológico). Nunca nulo nesta tabela, pois já filtrada.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN PRESENTE_NO_NTEP COMMENT 'Se a categoria acima está de fato prevista no Decreto 6.042/2007 (true) ou é agravo incluído só como referência comparativa (false: Chikungunya, Zika, Intoxicação Exógena). Derivado diretamente da categoria (fonte única de verdade), nunca calculado em paralelo a ela.';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN CAPITULO_CID_OFICIAL COMMENT 'Capítulo da CID-10 (I a XXII) calculado a partir do código CID real da notificação, seguindo a estrutura oficial da OMS (não herdado do texto do Decreto).';
ALTER TABLE workspace.sinan.gold_sinan_ntep_detalhe ALTER COLUMN REGIAO_NOTIF COMMENT 'Região geográfica (Norte/Nordeste/Sudeste/Sul/Centro-Oeste) derivada do primeiro dígito do código IBGE do município de notificação.';


COMMENT ON TABLE workspace.sinan.dim_notificante IS
'Dimensão (esquema estrela): combinações distintas de sexo e raça/cor observadas nas notificações da gold_sinan_ntep_detalhe. Chave: SK_NOTIFICANTE.';

ALTER TABLE workspace.sinan.dim_notificante ALTER COLUMN SK_NOTIFICANTE COMMENT 'Chave substituta (surrogate key) da dimensão, gerada por row_number() - não corresponde a nenhum identificador do SINAN.';
ALTER TABLE workspace.sinan.dim_notificante ALTER COLUMN SEXO COMMENT 'Sexo do paciente: Masculino, Feminino ou Ignorado.';
ALTER TABLE workspace.sinan.dim_notificante ALTER COLUMN RACA COMMENT 'Raça/cor do paciente: Branca, Preta, Amarela, Parda, Indígena ou Ignorado.';


COMMENT ON TABLE workspace.sinan.dim_local_notificacao IS
'Dimensão (esquema estrela): combinações distintas de região, UF e município de notificação observadas na gold_sinan_ntep_detalhe. Município de residência não faz parte desta dimensão. Chave: SK_LOCAL.';

ALTER TABLE workspace.sinan.dim_local_notificacao ALTER COLUMN SK_LOCAL COMMENT 'Chave substituta (surrogate key) da dimensão, gerada por row_number() - não corresponde a nenhum identificador do SINAN.';
ALTER TABLE workspace.sinan.dim_local_notificacao ALTER COLUMN REGIAO_NOTIF COMMENT 'Região geográfica de notificação (Norte/Nordeste/Sudeste/Sul/Centro-Oeste).';
ALTER TABLE workspace.sinan.dim_local_notificacao ALTER COLUMN UF_NOTIF COMMENT 'Código da UF de notificação (dois primeiros dígitos do código IBGE do município de notificação).';
ALTER TABLE workspace.sinan.dim_local_notificacao ALTER COLUMN COD_MUN_NOTIF COMMENT 'Código IBGE completo (6 dígitos) do município de notificação.';


COMMENT ON TABLE workspace.sinan.dim_notificacao IS
'Dimensão (esquema estrela): combinações distintas de ano, CID, capítulo CID oficial, agravo e classificação NTEP observadas na gold_sinan_ntep_detalhe. Chave: SK_NOTIFICACAO.';

ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN SK_NOTIFICACAO COMMENT 'Chave substituta (surrogate key) da dimensão, gerada por row_number() - não corresponde a nenhum identificador do SINAN (o SINAN não expõe número de notificação nos extratos públicos do FTP).';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN ANO_NOTIF COMMENT 'Ano da notificação.';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN CID10 COMMENT 'Código CID-10 da notificação, maiúsculo.';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN CAPITULO_CID_OFICIAL COMMENT 'Capítulo da CID-10 (I a XXII) calculado a partir do CID real, conforme estrutura oficial da OMS.';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN AGRAVO COMMENT 'Nome do agravo/doença conforme classificado neste projeto (ex.: DENGUE, ACIDTRAB).';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN CATEGORIA_NTEP COMMENT 'Categoria do NTEP a que o CID/agravo desta notificação corresponde.';
ALTER TABLE workspace.sinan.dim_notificacao ALTER COLUMN PRESENTE_NO_NTEP COMMENT 'Se a categoria está de fato prevista no Decreto 6.042/2007 (true) ou é agravo incluído só como referência comparativa (false).';


COMMENT ON TABLE workspace.sinan.fato_notificacoes_ntep IS
'Fato (esquema estrela): contagem de notificações agregada por combinação de SK_NOTIFICANTE, SK_LOCAL e SK_NOTIFICACAO, a partir da gold_sinan_ntep_detalhe (grão de notificação individual). Substitui os antigos agregados gold_sinan_nivel_brasil/regiao/uf - qualquer corte geográfico ou demográfico passa a ser obtido via JOIN + GROUP BY com as dimensões.';

ALTER TABLE workspace.sinan.fato_notificacoes_ntep ALTER COLUMN SK_FATO COMMENT 'Chave substituta (surrogate key) da própria fato, gerada por row_number() - necessária porque o SINAN não expõe um identificador de notificação nos extratos públicos, e a fato é agregada (uma linha não corresponde a uma única notificação).';
ALTER TABLE workspace.sinan.fato_notificacoes_ntep ALTER COLUMN SK_NOTIFICANTE COMMENT 'Chave estrangeira para dim_notificante (sexo, raça).';
ALTER TABLE workspace.sinan.fato_notificacoes_ntep ALTER COLUMN SK_LOCAL COMMENT 'Chave estrangeira para dim_local_notificacao (região, UF, município de notificação).';
ALTER TABLE workspace.sinan.fato_notificacoes_ntep ALTER COLUMN SK_NOTIFICACAO COMMENT 'Chave estrangeira para dim_notificacao (ano, CID, capítulo, agravo, categoria NTEP).';
ALTER TABLE workspace.sinan.fato_notificacoes_ntep ALTER COLUMN N_NOTIFICACOES COMMENT 'Contagem de notificações da gold_sinan_ntep_detalhe que compartilham a mesma combinação das três chaves estrangeiras.';